# 七零大院：冷面糙汉宠妻成瘾

**Runtime → Run all** — ใช้เวลา ~2-3 นาที แล้วไฟล์โหลดอัตโนมัติ

In [ ]:
!pip install -q requests beautifulsoup4

In [ ]:
import re, time, requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files

BASE     = 'https://ixdzs8.com/read/609163'
OUTPUT   = '七零大院_冷面糙汉宠妻成瘾.txt'
WORKERS  = 10
MAX_CH   = 490  # probe up to this number

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-CN,zh;q=0.9',
    'Referer': 'https://ixdzs8.com/',
}
SKIP = re.compile(r'(ixdzs8|请记住本书首发域名|www\.|m\.ixdzs8|本章未完)', re.I)

def fetch_chapter(n):
    url = f'{BASE}/p{n}.html'
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code == 404:
            return n, None, None
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, 'html.parser')
        # title
        t = soup.select_one('h1,.chapter-title,.title')
        title = t.get_text(strip=True) if t else f'第{n}章'
        # content
        el = (soup.select_one('#page') or soup.select_one('.read-content')
              or soup.select_one('.content') or soup.select_one('article'))
        if not el:
            return n, title, ''
        for tag in el.select('script,style,.ad,ins'): tag.decompose()
        lines = [p.get_text(' ', strip=True) for p in el.find_all(['p','br'])]
        if not lines:
            lines = [l.strip() for l in el.get_text('\n').splitlines() if l.strip()]
        content = '\n'.join(l for l in lines if not SKIP.search(l))
        return n, title, content
    except Exception as e:
        return n, f'第{n}章', f'[error: {e}]'

print(f'กำลังดาวน์โหลดด้วย {WORKERS} workers พร้อมกัน…\n')
results = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(fetch_chapter, n): n for n in range(1, MAX_CH + 1)}
    done = 0
    for f in as_completed(futures):
        n, title, content = f.result()
        if title is not None:
            results[n] = (title, content)
        done += 1
        if done % 50 == 0:
            print(f'  {done}/{MAX_CH} เสร็จแล้ว ({len(results)} บทพบ)')

chapters = sorted((n, t, c) for n, (t, c) in results.items())
print(f'\nรวม {len(chapters)} บท — กำลังเขียนไฟล์…')

with open(OUTPUT, 'w', encoding='utf-8') as f:
    for n, title, content in chapters:
        f.write(f'\n\n第{n}章　{title}\n\n{content}\n')

print('✓ เสร็จ! กำลังโหลดไฟล์…')
files.download(OUTPUT)